# Misja 5: kolorowy kurier i punkt odbioru paczek 🎨

**Cel:** wysłać żądanie, odebrać odpowiedź i napisać własny callback serwera.
Pracujemy z turtlesim; notebook uruchamia go sam. Zamknij teleop i inne panele sterowania.

```text
klient ── żądanie SetPen ──► /turtle1/set_pen ──► serwer w turtlesim
klient ◄── odpowiedź ────────────────────────────┘
```

Topic dostarcza strumień; **service** odpowiada na konkretne żądanie.
Jeden serwer może obsługiwać wielu klientów. Każde wywołanie ma własną odpowiedź.
Usługi pasują do krótkich operacji; długi przejazd z postępem będzie action w dalszym kursie.

**Komórka startowa poniżej jest gotowa — nie musisz jej zmieniać.**
`lab` pilnuje uruchomienia, czasu oczekiwania i sprzątania. `node` to Twój uczestnik ROS,
na którym za chwilę utworzymy interfejsy komunikacji. `globals()` przekazuje helperowi stan
notebooka, żeby ponowny start mógł usunąć stare obiekty.

In [ ]:
from intro_ros import notebook_lab

# Gotowe zaplecze: tworzy node i sprząta poprzednią sesję przy ponownym starcie.
lab = notebook_lab('odbior_paczek', globals())
node = lab.node
lab.start_turtlesim()

## 1. Odkryj usługę i jej kontrakt

W terminalu kontenera:

```bash
ros2 service list -t
ros2 service type /turtle1/set_pen
ros2 interface show turtlesim/srv/SetPen
```

Albo z Pythona:

In [ ]:
lab.spin_for(0.5)
for name, types in node.get_service_names_and_types():
    if name.startswith('/turtle1/') or name in ('/clear', '/reset'):
        print(name, types)

`SetPen` ma pola żądania: `r`, `g`, `b`, `width`, `off` (liczby całkowite).
`---` w definicji oddziela **Request** od **Response**. Odpowiedź tego serwisu jest pusta;
pomyślnie ukończony future informuje o odpowiedzi, a zmiana koloru będzie widoczna przy kolejnym ruchu.
Inne serwisy mają dane w odpowiedzi — zrobimy taki za chwilę.

**Przewidź:** czy `off=1` zatrzyma robota czy tylko wyłączy rysowanie?

## 2. Wywołanie krok po kroku: client, request, future, spin

**Future** to obiekt „wynik będzie później”. `call_async` wysyła żądanie i od razu wraca;
nie oznacza, że serwer już odpowiedział. Executor musi obsłużyć odbiór odpowiedzi.
Maksymalnie 5 s czekamy na serwer, a potem maksymalnie 5 s na wynik.

In [ ]:
from turtlesim.srv import SetPen

client = node.create_client(SetPen, '/turtle1/set_pen')
try:
    if not client.wait_for_service(timeout_sec=5.0):
        raise TimeoutError('Nie widzę set_pen. Sprawdź, czy turtlesim działa.')
    request = SetPen.Request(r=40, g=220, b=80, width=4, off=0)
    future = client.call_async(request)
    print('Żądanie wysłane. Czy odpowiedź już jest?', future.done())
    lab.wait_for(future.done, timeout=5, description='odpowiedź set_pen')
    print('Odpowiedź:', future.result())
finally:
    if 'future' in globals() and not future.done():
        client.remove_pending_request(future)
        future.cancel()
    node.destroy_client(client)

**Oczekiwany wynik:** wydruk pustej odpowiedzi `SetPen_Response()` bez błędu.
Narysujmy łuk, aby zobaczyć zielony pisak. Pamiętaj: odpowiedź set_pen nie oznacza,
że robot się poruszył — zmiana pisaka i jazda to dwie osobne operacje.

In [ ]:
from geometry_msgs.msg import Twist

if 'ruch' not in globals():
    ruch = node.create_publisher(Twist, '/turtle1/cmd_vel', 10)
lab.watch_motion(ruch)
lab.drive(ruch, linear=1.0, angular=1.0, seconds=1.0)

## 3. Minigra: pocztówka w trzech kolorach

Użyj trzech kolorów RGB (0–255), zmieniając pisak przed każdym łukiem.
`lab.call` skraca dokładnie sekwencję z poprzedniej komórki: tworzy klienta,
czeka na serwer, wysyła request, spinuje do odpowiedzi i usuwa klienta.
Ma wspólny limit czasu na połączenie i odpowiedź. **Timeout nie cofa operacji serwera**:
nie ponawiaj w ciemno operacji z efektem ubocznym.

Zmień kolory, szerokości i kierunki, żeby stworzyć własny znak kuriera.

In [ ]:
kolory = [(240, 60, 80), (60, 120, 250), (240, 220, 40)]
for r, g, b in kolory:
    lab.call(SetPen, '/turtle1/set_pen', SetPen.Request(r=r, g=g, b=b, width=4, off=0))
    lab.drive(ruch, linear=1.0, angular=1.8, seconds=0.6)

## 4. Teleport: zmieniamy świat, nie prędkość

`TeleportAbsolute` ustawia pozycję i kierunek natychmiast. `theta` jest w radianach.
Wyłączamy pisak na czas teleportu, żeby nie rysować kreski między miejscami.
**Przewidź:** co zobaczy subscriber `/turtle1/pose` po tym wywołaniu?

In [ ]:
from turtlesim.srv import TeleportAbsolute

lab.call(SetPen, '/turtle1/set_pen', SetPen.Request(off=1))
lab.call(TeleportAbsolute, '/turtle1/teleport_absolute',
         TeleportAbsolute.Request(x=5.0, y=5.0, theta=0.0))
lab.call(SetPen, '/turtle1/set_pen', SetPen.Request(r=40, g=220, b=80, width=3, off=0))

## 5. Zbuduj własny serwer: „czy paczka dotarła?”

Punkt odbioru jest w **(7,5)**, w promieniu 0.5 jednostki.
Subscriber przechowuje pozycję, a usługa `/kurier/odbior` **ocenia tę pozycję dopiero na żądanie**.
Typ `std_srvs/srv/Trigger` ma pusty request i odpowiedź z `success` oraz `message`.
`success=False` jest poprawną odpowiedzią gry, a nie awarią transportu ROS.

Zaimplementuj regułę `ocen_dostawe`: za daleko → False, w punkcie → True.
Testujemy też brak pozycji. Nie otrzymaliśmy danych? Nie wolno potwierdzać dostawy.

In [ ]:
from turtlesim.msg import Pose
import math
import time

if 'pose_subscription' in globals():
    node.destroy_subscription(pose_subscription)
stan = {'pose': None, 'time': None}

def odbierz_pozycje(msg):
    stan.update(pose=msg, time=time.monotonic())

pose_subscription = node.create_subscription(Pose, '/turtle1/pose', odbierz_pozycje, 10)
lab.wait_for(lambda: stan['pose'] is not None, description='pozycja żółwia')

In [ ]:
CEL = (7.0, 5.0)

def ocen_dostawe(pose):
    if pose is None:
        return False, 'Nie mam pozycji robota'
    return False, 'Jeszcze nie sprawdzam odległości'  # TODO: sprawdź promień 0.5

proby = [ocen_dostawe(None)[0], ocen_dostawe(Pose(x=5.0, y=5.0))[0],
         ocen_dostawe(Pose(x=7.0, y=5.0))[0], ocen_dostawe(Pose(x=7.3, y=5.0))[0]]
dostawa_gotowa = proby == [False, False, True, True]
print('✓ Punkt odbioru gotowy!' if dostawa_gotowa else f'Jeszcze krok: {proby}; oczekuję [False, False, True, True].')

Serwer otrzymuje obiekty `request` i `response`; musi zwrócić wypełnioną odpowiedź.
Callback powinien być krótki. Nie uruchamiamy w nim blokującego oczekiwania na inną usługę.
Tutaj sprawdzamy tylko ostatnio odebrany stan, jego wiek i numer potwierdzenia.

In [ ]:
from std_srvs.srv import Trigger

if 'server' in globals():
    node.destroy_service(server)
bilet = {'wydany': False, 'numer': 0}

def odbior_paczki(request, response):
    if not dostawa_gotowa:
        response.success = False
        response.message = 'Najpierw ukończ test ocen_dostawe'
    elif stan['time'] is None or time.monotonic() - stan['time'] > 0.5:
        response.success = False
        response.message = 'Pozycja jest nieznana lub nieaktualna'
    elif bilet['wydany']:
        response.success = False
        response.message = 'Ta paczka już odebrana! Nie naliczam punktu drugi raz'
    else:
        response.success, response.message = ocen_dostawe(stan['pose'])
        if response.success:
            bilet['wydany'] = True
            bilet['numer'] += 1
            response.message = f'Paczka odebrana! Potwierdzenie #{bilet["numer"]}'
    return response

server = node.create_service(Trigger, '/kurier/odbior', odbior_paczki)
print('Serwer /kurier/odbior gotowy.')

## 6. Finał: jedna paczka, jedno potwierdzenie

Klient i serwer są tutaj w tym samym notebooku, żeby nie potrzebować drugiego kernela;
w ROS nadal komunikują się przez usługę. Ten sam serwer może obsługiwać zewnętrznego klienta.

1. Zresetuj pozycję do (5,5). Zapytaj — jeszcze za daleko.
2. Jedź naprzód przez około 2 s z prędkością 1. Zapytaj — paczka powinna być w punkcie.
3. Zapytaj ponownie — nie dostaniesz drugiego punktu za tę samą paczkę.

**Wynik zależy od faktycznej pozycji**, nie od liczby sekund jazdy. Jeśli jesteś poza
promieniem, sprawdź `stan['pose']` i popraw pozycję krótkim ruchem.
Uzupełnij regułę z części 5 przed rozpoczęciem finału.

In [ ]:
if dostawa_gotowa:
    bilet['wydany'] = False
    lab.call(TeleportAbsolute, '/turtle1/teleport_absolute',
             TeleportAbsolute.Request(x=5.0, y=5.0, theta=0.0))
    lab.spin_for(0.3)  # odbierz aktualizację pozycji po teleportacji
    odpowiedz_przed = lab.call(Trigger, '/kurier/odbior', Trigger.Request())
    print('Przed jazdą:', odpowiedz_przed.success, odpowiedz_przed.message)
    lab.drive(ruch, linear=1.0, seconds=2.0)
    odpowiedz_po = lab.call(Trigger, '/kurier/odbior', Trigger.Request())
    print('Po jeździe:', odpowiedz_po.success, odpowiedz_po.message)
    odpowiedz_druga = lab.call(Trigger, '/kurier/odbior', Trigger.Request())
    print('Jeszcze raz:', odpowiedz_druga.success, odpowiedz_druga.message)
else:
    print('Robot czeka: ukończ ocen_dostawe i ponów komórkę serwera.')

**Bonus: klient z terminala.** Uruchom komórkę poniżej, a w ciągu 15 s wywołaj
`ros2 service call /kurier/odbior std_srvs/srv/Trigger '{}'` z terminala kontenera.
Bez spinowania serwer jest widoczny w grafie, ale nie obsłuży requestu.

In [ ]:
# Zmień na True, gdy terminal klienta jest gotowy.
KLIENT_Z_TERMINALA = False
if KLIENT_Z_TERMINALA:
    lab.spin_for(15.0)

## Co teraz rozumiesz?

Wyjaśnij partnerowi różnicę między: opublikowaniem prędkości, odczytem położenia
oraz potwierdzeniem dostawy. Która część może odmówić z powodów reguł gry?
Która potrzebuje spin, żeby callback się wykonał?

**Bonus detektywa:** znajdź `/turtlesim/get_parameters`, sprawdź jego typ i poproś o
`use_sim_time` używając `GetParameters.Request(names=['use_sim_time'])`.
Zajrzyj do `response.values[0].bool_value`. Sam odczyt konfiguracji nie zmienia robota.
[Dokumentacja Jazzy: usługi](https://docs.ros.org/en/jazzy/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Services/Understanding-ROS2-Services.html).

## Zapisz odkrycie i posprzątaj

Zapisz jednym zdaniem: co wysyłał Twój node i co działo się dopiero podczas spinowania?
Uruchom tę komórkę przed przejściem dalej. Zatrzyma ruch, usunie node i zamknie
turtlesim, **jeśli ten notebook go uruchomił**. Kolejny notebook uruchomi swój.
Przy przerwaniu komórki też przejdź tutaj; po restarcie kernela zamknij pozostawione okno turtlesim ręcznie.

In [ ]:
lab.close()

Gratulacje — połączyłeś komendy, pomiar stanu i odpowiedź na żądanie w jednym robocie!
[Dalej: 6. Nawigacja z Nav2 →](6.%20Navigate%20-%20nav2.ipynb)